# Free-Energy-Based Predictive Stability and Secure Resource Orchestration for Hybrid QKD–PQC Satellite–Terrestrial Networks

Reproducible simulation notebook. It builds the role-aware satellite–terrestrial network, simulates the dynamic link process, computes the hybrid QKD–PQC security quantities (Q_p, C_p, S_p, H_p, risk, free energy), the system-level indices (SCI, SDI) and runs all allocation methods, then produces every table and figure of the article: reference run, method comparison, ablation, repeated seeds, stress scenarios, parameter sensitivity and the prospective evaluation of the SCI.

Run all cells top to bottom. Outputs are written to local disk (`/content/Outputs/quantum_secure_comm` in Colab). The full run with 30 seeds takes about 15 minutes on a single CPU; set the environment variable `QSCI_SEEDS` to a smaller value for a quick check.

In [1]:
# ---- Runtime setup ------------------------------------------------------------
# Outputs are written to local disk. In Colab this is /content (never Google Drive);
# set BACKUP_TO_DRIVE = True to copy the small tables/figures to MyDrive/Outputs/<project> at the end.
import os, sys, types, shutil
PROJECT_NAME = "quantum_secure_comm"
BACKUP_TO_DRIVE = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
OUTPUT_ROOT = f"/content/Outputs/{PROJECT_NAME}" if IN_COLAB else f"./Outputs/{PROJECT_NAME}"
N_SEEDS_ENV = os.environ.get("QSCI_SEEDS", "30")
print("Output root:", OUTPUT_ROOT)


Output root: ./Outputs/quantum_secure_comm


## 1. Model
Topology, link-state process, surrogate QKD/PQC rates, path-level security quantities, system-level indices and allocation methods.

In [2]:
# =============================================================================
# Free-Energy-Based Predictive Stability and Secure Resource Orchestration
# for Hybrid QKD-PQC Satellite-Terrestrial Networks  --  simulation model
# =============================================================================
# This module defines the role-aware STIN, the dynamic link-state process, the
# hybrid QKD-PQC security quantities (Q_p, C_p, S_p, H_p, risk, free energy),
# the system-level indices (SCI, SDI) and the allocation methods.  Everything
# is a pure function of (seed, scenario, parameters) so that repeated runs,
# stress scenarios, sensitivity sweeps and ablations are reproducible.
# =============================================================================
import math
import hashlib
from copy import deepcopy

import numpy as np
import pandas as pd
import networkx as nx

# -----------------------------------------------------------------------------
# Nominal parameters (Table 2 of the manuscript)
# -----------------------------------------------------------------------------
PARAMS = {
    "time_horizon": 24,
    "max_candidate_paths_per_flow": 8,
    "max_path_hops": 6,
    # hybrid security level S_p
    "alpha": 0.55,            # QKD weight
    "beta_trust": 0.22,       # trust-penalty weight
    # free energy F_p = (1 - S_p) + lambda_H * H_p + lambda_R * R_p
    "lambda_H": 0.25,
    "lambda_R": 0.45,
    # allocation utility U_p = -F_p - w_ell * c_p
    "w_ell": 0.30,
    # path risk weights (trust, noise, qber)
    "beta_risk": (0.45, 0.30, 0.25),
    "noise_max": 0.12,
    "qber_max": 0.15,
    # QKD / PQC surrogate constants
    "reconciliation_efficiency": 0.92,
    "pqc_overhead": 0.12,
    # reference rates normalising Q_p and C_p: kappa x the largest flow key / traffic demand (kappa = 2)
    "kappa": 2.0,
    "q_ref": 2.0 * 0.040,
    "b_ref": 2.0 * 5.0,
    # regime thresholds (set from m_ref, see regime_thresholds)
    "m_ref": 0.20,
    "secure_fraction": 0.80,
    "sparse_fraction": 0.40,
    # ablation switches
    "use_quantum_signal": True,
    "use_risk_penalty": True,
    "use_trust_signal": True,
    "use_entropy_term": True,
}

# -----------------------------------------------------------------------------
# Topology
# -----------------------------------------------------------------------------
NODES = ["SAT_1", "SAT_2", "UAV_1", "UAV_2", "GW_1", "GW_2", "BS_1", "BS_2", "BS_3", "DC_1"]
NODE_POS = {
    "SAT_1": (0.12, 0.92), "SAT_2": (0.88, 0.92),
    "UAV_1": (0.24, 0.70), "UAV_2": (0.76, 0.70),
    "GW_1": (0.24, 0.48), "GW_2": (0.76, 0.48),
    "BS_1": (0.12, 0.20), "BS_2": (0.50, 0.20), "BS_3": (0.88, 0.20),
    "DC_1": (0.50, 0.04),
}
EDGES = [
    ("SAT_1", "SAT_2"), ("SAT_1", "UAV_1"), ("SAT_2", "UAV_2"), ("SAT_1", "GW_1"), ("SAT_2", "GW_2"),
    ("UAV_1", "GW_1"), ("UAV_2", "GW_2"), ("UAV_1", "BS_2"), ("UAV_2", "BS_2"),
    ("GW_1", "BS_1"), ("GW_1", "BS_2"), ("GW_2", "BS_2"), ("GW_2", "BS_3"),
    ("BS_1", "BS_2"), ("BS_2", "BS_3"), ("BS_1", "DC_1"), ("BS_2", "DC_1"), ("BS_3", "DC_1"),
]


def node_role(name):
    if name.startswith("SAT"): return "satellite"
    if name.startswith("UAV"): return "aerial"
    if name.startswith("GW"): return "gateway"
    if name.startswith("BS"): return "base_station"
    if name.startswith("DC"): return "edge_cloud"
    return "unknown"


def node_layer(name):
    r = node_role(name)
    if r == "satellite": return "space"
    if r == "aerial": return "air"
    return "ground"


def edge_role_class(u, v):
    ru, rv = node_role(u), node_role(v)
    pair = {ru, rv}
    if pair == {"satellite"}: return "inter_satellite"
    if pair == {"satellite", "gateway"}: return "satellite_gateway"
    if pair == {"satellite", "aerial"}: return "satellite_aerial"
    if pair == {"aerial", "gateway"}: return "aerial_gateway"
    if pair == {"aerial", "base_station"}: return "aerial_access"
    if pair == {"gateway", "base_station"}: return "gateway_backhaul"
    if pair == {"base_station"}: return "terrestrial_backhaul"
    if pair == {"base_station", "edge_cloud"}: return "metro_core"
    if pair == {"gateway", "edge_cloud"}: return "gateway_core"
    return "heterogeneous_ground"


LINK_CLASS_PARAMS = {
    "inter_satellite":     dict(transmissivity=(0.20, 0.38), noise=(0.010, 0.030), qber=(0.010, 0.040), latency_ms=(12, 22), capacity=(8, 16),  assurance=(0.82, 0.96), trust=(0.82, 0.94)),
    "satellite_gateway":   dict(transmissivity=(0.14, 0.32), noise=(0.020, 0.060), qber=(0.020, 0.065), latency_ms=(20, 40), capacity=(6, 14),  assurance=(0.80, 0.95), trust=(0.72, 0.88)),
    "satellite_aerial":    dict(transmissivity=(0.16, 0.36), noise=(0.018, 0.055), qber=(0.018, 0.060), latency_ms=(16, 32), capacity=(6, 14),  assurance=(0.80, 0.94), trust=(0.70, 0.86)),
    "aerial_gateway":      dict(transmissivity=(0.30, 0.58), noise=(0.010, 0.035), qber=(0.012, 0.045), latency_ms=(7, 15),   capacity=(10, 22), assurance=(0.82, 0.96), trust=(0.76, 0.90)),
    "aerial_access":       dict(transmissivity=(0.36, 0.66), noise=(0.008, 0.030), qber=(0.010, 0.040), latency_ms=(5, 12),   capacity=(12, 24), assurance=(0.84, 0.97), trust=(0.78, 0.92)),
    "gateway_backhaul":    dict(transmissivity=(0.62, 0.90), noise=(0.004, 0.018), qber=(0.006, 0.020), latency_ms=(2, 8),    capacity=(18, 36), assurance=(0.86, 0.98), trust=(0.84, 0.96)),
    "terrestrial_backhaul":dict(transmissivity=(0.72, 0.95), noise=(0.003, 0.015), qber=(0.005, 0.018), latency_ms=(1, 6),    capacity=(20, 40), assurance=(0.88, 0.99), trust=(0.86, 0.97)),
    "metro_core":          dict(transmissivity=(0.78, 0.96), noise=(0.002, 0.012), qber=(0.004, 0.016), latency_ms=(1, 5),    capacity=(24, 44), assurance=(0.90, 0.99), trust=(0.88, 0.98)),
    "gateway_core":        dict(transmissivity=(0.68, 0.92), noise=(0.003, 0.015), qber=(0.005, 0.018), latency_ms=(2, 7),    capacity=(22, 40), assurance=(0.88, 0.99), trust=(0.86, 0.97)),
    "heterogeneous_ground":dict(transmissivity=(0.60, 0.88), noise=(0.004, 0.020), qber=(0.006, 0.022), latency_ms=(2, 8),    capacity=(16, 32), assurance=(0.84, 0.97), trust=(0.82, 0.95)),
}
QKD_CAPABLE = {"inter_satellite", "satellite_gateway", "satellite_aerial", "aerial_gateway", "aerial_access"}
SPACE_AIR = QKD_CAPABLE

FLOWS = [
    dict(flow_id="F1", src="SAT_1", dst="DC_1", mission="satellite_telemetry_backhaul",     security_req=0.58, priority=1.00, traffic_demand=5.0, key_demand=0.040),
    dict(flow_id="F2", src="SAT_2", dst="BS_1", mission="secure_satellite_access_delivery", security_req=0.52, priority=0.94, traffic_demand=4.5, key_demand=0.032),
    dict(flow_id="F3", src="UAV_1", dst="DC_1", mission="uav_control_and_data_return",      security_req=0.48, priority=0.88, traffic_demand=5.0, key_demand=0.024),
    dict(flow_id="F4", src="GW_2",  dst="BS_1", mission="gateway_to_edge_service_sync",     security_req=0.44, priority=0.78, traffic_demand=4.0, key_demand=0.016),
    dict(flow_id="F5", src="UAV_2", dst="BS_3", mission="uav_to_local_edge_delivery",       security_req=0.46, priority=0.82, traffic_demand=3.5, key_demand=0.016),
]

# -----------------------------------------------------------------------------
# Stress scenarios (Section 4.9)
# -----------------------------------------------------------------------------
SCENARIOS = {
    "nominal":           dict(qkd_space=1.00, qkd_ground=1.00, pqc_space=1.00, pqc_ground=1.00, lat_space=1.00, lat_ground=1.00, trust_shift=0.00, noise=1.00, qber=1.00, availability=1.00, traffic=1.00, key=1.00),
    "qkd_scarcity":      dict(qkd_space=0.55, qkd_ground=0.85, pqc_space=1.00, pqc_ground=1.00, lat_space=1.00, lat_ground=1.00, trust_shift=0.00, noise=1.00, qber=1.00, availability=0.95, traffic=1.00, key=1.15),
    "congestion":        dict(qkd_space=1.00, qkd_ground=1.00, pqc_space=0.82, pqc_ground=0.72, lat_space=1.12, lat_ground=1.30, trust_shift=0.00, noise=1.00, qber=1.00, availability=1.00, traffic=1.00, key=1.00),
    "trust_degradation": dict(qkd_space=1.00, qkd_ground=1.00, pqc_space=1.00, pqc_ground=1.00, lat_space=1.00, lat_ground=1.00, trust_shift=-0.15, noise=1.00, qber=1.00, availability=1.00, traffic=1.00, key=1.00),
    "link_noise":        dict(qkd_space=1.00, qkd_ground=1.00, pqc_space=1.00, pqc_ground=1.00, lat_space=1.00, lat_ground=1.00, trust_shift=0.00, noise=1.40, qber=1.35, availability=0.95, traffic=1.00, key=1.00),
    "traffic_load":      dict(qkd_space=1.00, qkd_ground=1.00, pqc_space=1.00, pqc_ground=1.00, lat_space=1.05, lat_ground=1.10, trust_shift=0.00, noise=1.00, qber=1.00, availability=1.00, traffic=1.30, key=1.20),
}

METHODS = ["proposed_hybrid", "latency_first", "pqc_trust", "qkd_priority", "max_margin_greedy", "shortest_path_pqc", "random"]
METHOD_LABELS = {
    "proposed_hybrid": "Proposed hybrid", "latency_first": "Latency-first", "pqc_trust": "PQC-trust",
    "qkd_priority": "QKD-priority", "max_margin_greedy": "Max-margin greedy", "shortest_path_pqc": "Shortest-path PQC",
    "random": "Random admissible",
}


# -----------------------------------------------------------------------------
# Elementary functions
# -----------------------------------------------------------------------------
def clip01(x):
    return float(np.clip(x, 0.0, 1.0))


def binary_entropy(p, eps=1e-12):
    p = float(np.clip(p, eps, 1.0 - eps))
    return float(-(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p)))


def qkd_rate(transmissivity, excess_noise, qber, reconciliation_efficiency=0.92):
    """Secret-key-rate surrogate, Eq. (16): PLOB-type loss scaling x BB84 secret fraction x noise penalty."""
    eta = float(np.clip(transmissivity, 1e-6, 0.999999))
    xi = max(0.0, float(excess_noise))
    q = float(np.clip(qber, 1e-6, 0.49))
    plob = -np.log2(1.0 - eta)
    secret_fraction = max(0.0, reconciliation_efficiency * (1.0 - 2.0 * binary_entropy(q)))
    return float(max(0.0, plob * secret_fraction * math.exp(-4.0 * xi)))


def pqc_secure_rate(link_capacity, assurance, pqc_overhead=0.12):
    """Overhead-adjusted PQC secure throughput, Eq. (18) numerator."""
    return float(max(0.0, link_capacity * (1.0 - pqc_overhead) * clip01(assurance)))


def communication_cost(latency_ms, hop_count, norm_latency=150.0, norm_hops=6):
    return float(0.70 * clip01(latency_ms / norm_latency) + 0.30 * clip01(hop_count / norm_hops))


# -----------------------------------------------------------------------------
# Topology and link-state generation
# -----------------------------------------------------------------------------
def build_graph(rng):
    G = nx.Graph()
    G.add_nodes_from(NODES)
    for n in G.nodes():
        G.nodes[n]["role"] = node_role(n)
        G.nodes[n]["layer"] = node_layer(n)
        G.nodes[n]["pos"] = NODE_POS[n]
    G.add_edges_from(EDGES)
    for u, v in G.edges():
        lc = edge_role_class(u, v)
        p = LINK_CLASS_PARAMS[lc]
        d = G[u][v]
        d["link_class"] = lc
        d["base_transmissivity"] = float(rng.uniform(*p["transmissivity"]))
        d["base_noise"] = float(rng.uniform(*p["noise"]))
        d["base_qber"] = float(rng.uniform(*p["qber"]))
        d["latency_ms"] = float(rng.uniform(*p["latency_ms"]))
        d["link_capacity"] = float(rng.uniform(*p["capacity"]))
        d["assurance"] = float(rng.uniform(*p["assurance"]))
        d["trust_score"] = float(rng.uniform(*p["trust"]))
        d["supports_qkd"] = lc in QKD_CAPABLE
    return G


def _dynamic_factors(link_class):
    if link_class in {"inter_satellite", "satellite_gateway", "satellite_aerial"}:
        return dict(weather=1.00, congestion=0.20, availability=0.94)
    if link_class in {"aerial_gateway", "aerial_access"}:
        return dict(weather=0.65, congestion=0.35, availability=0.96)
    return dict(weather=0.15, congestion=0.85, availability=0.985)


def simulate_link_states(G, rng, params):
    T = params["time_horizon"]
    t_idx = np.arange(T)
    weather = 0.5 + 0.5 * np.sin(2 * np.pi * t_idx / T + 0.6)
    congestion = 0.5 + 0.5 * np.sin(2 * np.pi * t_idx / T + 2.1)
    adversarial = 0.5 + 0.5 * np.sin(2 * np.pi * t_idx / T + 4.0)
    profile = {}
    for u, v in G.edges():
        profile[(u, v)] = dict(
            phase=rng.uniform(0, 2 * np.pi), trans_vol=rng.uniform(0.03, 0.08), noise_vol=rng.uniform(0.003, 0.012),
            qber_vol=rng.uniform(0.003, 0.010), cap_vol=rng.uniform(0.04, 0.12), lat_vol=rng.uniform(0.05, 0.18),
            trust_vol=rng.uniform(0.01, 0.05))
    rows = []
    for t in range(T):
        w_t, c_t, a_t = float(weather[t]), float(congestion[t]), float(adversarial[t])
        for u, v, d in G.edges(data=True):
            pr, f = profile[(u, v)], _dynamic_factors(d["link_class"])
            wave = np.sin(2 * np.pi * t / T + pr["phase"])
            wave_s = np.cos(2 * np.pi * t / T + pr["phase"] / 2.0)
            avail_score = float(np.clip(f["availability"] - 0.10 * f["weather"] * w_t - 0.03 * pr["trans_vol"] + 0.04 * wave_s, 0.70, 0.995))
            available = bool(rng.random() < avail_score)
            trans = d["base_transmissivity"] * (1.0 + pr["trans_vol"] * wave - 0.18 * f["weather"] * w_t + rng.normal(0, 0.015))
            if not available: trans *= rng.uniform(0.05, 0.25)
            trans = float(np.clip(trans, 0.001, 0.98))
            noise = d["base_noise"] * (1.0 + pr["noise_vol"] * 12.0 * abs(wave_s) + 0.55 * f["weather"] * w_t + rng.normal(0, 0.05))
            if not available: noise *= rng.uniform(1.2, 1.8)
            noise = float(np.clip(noise, 0.001, 0.20))
            qber = d["base_qber"] + 0.10 * noise + 0.08 * (1.0 - trans) + pr["qber_vol"] * wave_s + rng.normal(0, 0.003)
            if not available: qber += rng.uniform(0.01, 0.04)
            qber = float(np.clip(qber, 0.001, 0.25))
            cap = d["link_capacity"] * (1.0 - 0.20 * f["congestion"] * c_t + pr["cap_vol"] * wave + rng.normal(0, 0.02))
            cap = float(np.clip(cap, 1.0, d["link_capacity"] * 1.15))
            lat = d["latency_ms"] * (1.0 + 0.35 * f["congestion"] * c_t + pr["lat_vol"] * abs(wave) + rng.normal(0, 0.03))
            if not available: lat *= rng.uniform(1.10, 1.35)
            lat = float(np.clip(lat, 0.5, 120.0))
            trust = d["trust_score"] - (0.05 * a_t + 0.03 * (not available) + pr["trust_vol"] * max(0.0, wave_s) + rng.normal(0, 0.01))
            trust = float(np.clip(trust, 0.45, 0.99))
            rows.append(dict(time=t, u=u, v=v, link_class=d["link_class"], is_available=available, transmissivity=trans,
                             excess_noise=noise, qber=qber, capacity=cap, assurance=d["assurance"], trust_score=trust,
                             latency_ms=lat, supports_qkd=d["supports_qkd"]))
    return pd.DataFrame(rows)


def apply_scenario(link_df, scenario_name, params):
    """Deterministic scenario perturbation of the nominal link process (same seeds, same events)."""
    cfg = SCENARIOS[scenario_name]
    out = link_df.copy()
    is_sa = out["link_class"].isin(SPACE_AIR).values
    out["trust_score"] = np.clip(out["trust_score"] + cfg["trust_shift"], 0.30, 0.99)
    out["excess_noise"] = np.clip(out["excess_noise"] * cfg["noise"], 0.001, 0.25)
    out["qber"] = np.clip(out["qber"] * cfg["qber"], 0.001, 0.30)
    out["latency_ms"] = np.clip(out["latency_ms"] * np.where(is_sa, cfg["lat_space"], cfg["lat_ground"]), 0.5, 180.0)
    if cfg["availability"] < 1.0:
        drop = []
        for r in out.itertuples():
            frag = 1.0 if r.link_class in SPACE_AIR else 0.6
            p_drop = (1.0 - cfg["availability"]) * frag
            h = int(hashlib.md5(f"{scenario_name}-{r.time}-{r.u}-{r.v}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
            drop.append(bool(r.is_available) and h < p_drop)
        drop = np.array(drop)
        out.loc[drop, "is_available"] = False
        out.loc[drop, "transmissivity"] = np.clip(out.loc[drop, "transmissivity"] * 0.20, 0.001, 0.98)
        out.loc[drop, "excess_noise"] = np.clip(out.loc[drop, "excess_noise"] * 1.40, 0.001, 0.25)
        out.loc[drop, "qber"] = np.clip(out.loc[drop, "qber"] + 0.03, 0.001, 0.30)
        out.loc[drop, "latency_ms"] = np.clip(out.loc[drop, "latency_ms"] * 1.15, 0.5, 180.0)
    # rates
    q = [qkd_rate(r.transmissivity, r.excess_noise, r.qber, params["reconciliation_efficiency"]) if r.supports_qkd else 0.0 for r in out.itertuples()]
    out["q_rate"] = np.array(q) * np.where(is_sa, cfg["qkd_space"], cfg["qkd_ground"])
    out["p_rate"] = np.array([pqc_secure_rate(r.capacity, r.assurance, params["pqc_overhead"]) for r in out.itertuples()]) * np.where(is_sa, cfg["pqc_space"], cfg["pqc_ground"])
    return out


def scenario_flows(scenario_name):
    cfg = SCENARIOS[scenario_name]
    fl = deepcopy(FLOWS)
    for f in fl:
        f["traffic_demand"] *= cfg["traffic"]
        f["key_demand"] *= cfg["key"]
    return fl


# -----------------------------------------------------------------------------
# Candidate paths
# -----------------------------------------------------------------------------
def _admissible(path, params):
    hops = len(path) - 1
    if hops > params["max_path_hops"]: return False
    layers = [node_layer(n) for n in path]
    if sum(1 for i in range(len(layers) - 1) if layers[i] != layers[i + 1]) > hops: return False
    if sum(1 for n in path if node_role(n) == "satellite") > 2: return False
    return True


def _preselect_score(G, path):
    attrs = [G[a][b] for a, b in zip(path[:-1], path[1:])]
    hops = len(path) - 1
    mean_trust = np.mean([d["trust_score"] for d in attrs])
    tot_lat = sum(d["latency_ms"] for d in attrs)
    qkd_frac = sum(1 for d in attrs if d["supports_qkd"]) / max(hops, 1)
    return 0.45 * mean_trust + 0.25 * qkd_frac - 0.30 * communication_cost(tot_lat, hops)


def candidate_paths(G, flows, params):
    out = {}
    for f in flows:
        raw = nx.all_simple_paths(G, f["src"], f["dst"], cutoff=params["max_path_hops"])
        adm = [p for p in raw if _admissible(p, params)]
        adm.sort(key=lambda p: _preselect_score(G, p), reverse=True)
        out[f["flow_id"]] = adm[: params["max_candidate_paths_per_flow"]]
    return out


# -----------------------------------------------------------------------------
# Path-level quantities (Section 3)
# -----------------------------------------------------------------------------
def risk_components(edges, params):
    """Mean over links of the three normalised exposure terms of Eq. (22)."""
    if not edges: return 1.0, 1.0, 1.0
    rt = float(np.mean([1.0 - clip01(m["trust_score"]) for m in edges]))
    rn = float(np.mean([clip01(m["excess_noise"] / params["noise_max"]) for m in edges]))
    rq = float(np.mean([clip01(m["qber"] / params["qber_max"]) for m in edges]))
    return rt, rn, rq


def path_risk(edges, params):
    b1, b2, b3 = params["beta_risk"]
    rt, rn, rq = risk_components(edges, params)
    return clip01(b1 * rt + b2 * rn + b3 * rq)


def hybrid_security_level(Q, C, trust, params):
    alpha = params["alpha"] if params["use_quantum_signal"] else 0.0
    beta = params["beta_trust"] if params["use_trust_signal"] else 0.0
    return clip01(alpha * Q + (1.0 - alpha) * C - beta * (1.0 - trust))


def free_energy(S, H, R, params):
    lH = params["lambda_H"] if params["use_entropy_term"] else 0.0
    lR = params["lambda_R"] if params["use_risk_penalty"] else 0.0
    return float((1.0 - S) + lH * H + lR * R)


def evaluate_path(path, t, lookup, residual, flow, params):
    edges = []
    available = True
    for a, b in zip(path[:-1], path[1:]):
        key = tuple(sorted((a, b)))
        row = lookup[(t, key)]
        res = residual[key]
        if not row["is_available"]: available = False
        edges.append(dict(row, q_budget=res["q_budget"], p_budget=res["p_budget"]))
    hops = len(path) - 1
    latency = float(sum(m["latency_ms"] for m in edges))
    risk = path_risk(edges, params)
    rt, rn, rq = risk_components(edges, params)
    degradation = float(1.0 - np.prod([np.clip(m["transmissivity"], 1e-6, 1.0) for m in edges]))
    trust = float(np.mean([m["trust_score"] for m in edges]))
    qber_path = float(max(m["qber"] for m in edges))
    H = binary_entropy(qber_path)
    qkd_edges = [m for m in edges if m["supports_qkd"]]
    qkd_frac = len(qkd_edges) / max(len(edges), 1)
    p_raw = min(m["p_budget"] for m in edges)
    q_seg = min(m["q_budget"] for m in qkd_edges) if qkd_edges else 0.0
    q_raw = qkd_frac * q_seg
    Q = clip01(q_raw / params["q_ref"])
    C = clip01(p_raw / params["b_ref"])
    q_share = float(Q / max(Q + C, 1e-9))
    S = hybrid_security_level(Q, C, trust, params)
    M = S - flow["security_req"]
    F = free_energy(S, H, risk, params)
    cost = communication_cost(latency, hops)
    U = -F - params["w_ell"] * cost
    service_ok = p_raw >= flow["traffic_demand"]
    req_key = flow["key_demand"] * max(qkd_frac, 0.35)
    key_ok = (q_seg >= req_key) if qkd_frac > 0 else True
    # Hybrid fallback (Section 3.6): an empty QKD key budget does not block the flow; the path falls
    # back to PQC-only protection, which lowers Q_p and hence S_p.  Only capacity blocks service.
    resource_ok = bool(available and service_ok)
    secure_ok = bool(resource_ok and S >= flow["security_req"])
    return dict(time=t, flow_id=flow["flow_id"], path=" -> ".join(path), hop_count=hops, security_req=flow["security_req"],
                priority=flow["priority"], path_available=available, resource_feasible=resource_ok, secure_feasible=secure_ok, key_feasible=bool(key_ok),
                Q=Q, C=C, q_share=q_share, trust_path=trust, latency_path=latency, risk_path=risk, degradation_path=degradation,
                qber_path=qber_path, H_path=H, risk_trust=rt, risk_noise=rn, risk_qber=rq, security_level=S, security_margin=M, free_energy=F, cost=cost, functional_score=U,
                qkd_edge_fraction=qkd_frac, p_raw=p_raw, q_raw=q_raw)


def consume(path_str, residual, flow, G):
    nodes = path_str.split(" -> ")
    el = list(zip(nodes[:-1], nodes[1:]))
    qkd_frac = np.mean([1.0 if G[a][b]["supports_qkd"] else 0.0 for a, b in el]) if el else 0.0
    for a, b in el:
        key = tuple(sorted((a, b)))
        residual[key]["p_budget"] = max(0.0, residual[key]["p_budget"] - flow["traffic_demand"])
        if G[a][b]["supports_qkd"]:
            residual[key]["q_budget"] = max(0.0, residual[key]["q_budget"] - flow["key_demand"] * max(qkd_frac, 0.35))


def blocked_row(flow, t, method):
    return dict(method=method, time=t, flow_id=flow["flow_id"], path="BLOCKED", hop_count=np.nan, security_req=flow["security_req"],
                priority=flow["priority"], path_available=False, resource_feasible=False, secure_feasible=False, key_feasible=False, Q=0.0, C=0.0,
                q_share=0.0, trust_path=0.0, latency_path=np.nan, risk_path=1.0, degradation_path=1.0, qber_path=np.nan, H_path=1.0,
                security_level=0.0, security_margin=-flow["security_req"], free_energy=np.nan, cost=np.nan, functional_score=np.nan,
                qkd_edge_fraction=0.0, p_raw=0.0, q_raw=0.0, selection_status="blocked", allocated=False)


# -----------------------------------------------------------------------------
# Allocation methods
# -----------------------------------------------------------------------------
def proposed_score(c, ctrl):
    """Allocation utility of the proposed controller, U_p = -F_p - w_ell c_p, evaluated with the
    controller parameters `ctrl` (ablations and sensitivity sweeps change ctrl, not the state model)."""
    Q = c["Q"] if ctrl["use_quantum_signal"] else 0.0
    trust = c["trust_path"] if ctrl["use_trust_signal"] else 1.0
    S = clip01(ctrl["alpha"] * Q + (1.0 - ctrl["alpha"]) * c["C"] - ctrl["beta_trust"] * (1.0 - trust))
    b1, b2, b3 = ctrl["beta_risk"]
    R = clip01(b1 * c["risk_trust"] + b2 * c["risk_noise"] + b3 * c["risk_qber"]) if ctrl["use_risk_penalty"] else 0.0
    H = c["H_path"] if ctrl["use_entropy_term"] else 0.0
    F = (1.0 - S) + ctrl["lambda_H"] * H + ctrl["lambda_R"] * R
    return float(-F - ctrl["w_ell"] * c["cost"])


def selection_score(c, method, rng, ctrl=None):
    if method == "proposed_hybrid":
        return proposed_score(c, ctrl) if ctrl is not None else c["functional_score"]
    if method == "latency_first":
        return -c["latency_path"] + 5.0 * c["trust_path"] - 2.0 * c["risk_path"]
    if method == "pqc_trust":
        return 0.60 * c["C"] + 0.25 * c["trust_path"] - 0.15 * c["cost"]
    if method == "qkd_priority":
        return 0.45 * c["Q"] + 0.25 * c["q_share"] + 0.15 * c["trust_path"] - 0.15 * c["cost"]
    if method == "max_margin_greedy":
        return c["security_margin"]
    if method == "shortest_path_pqc":
        return -c["latency_path"]
    if method == "random":
        return rng.random()
    raise ValueError(method)


def run_method(method, G, lookup, flows, cands, params, rng=None, ctrl=None):
    """Run one allocation method over the horizon. Returns a DataFrame with one row per (time, flow)."""
    rng = rng or np.random.default_rng(0)
    T = params["time_horizon"]
    order = sorted(flows, key=lambda x: (x["priority"], x["security_req"], x["traffic_demand"]), reverse=True)
    rows = []
    for t in range(T):
        residual = {tuple(sorted((u, v))): dict(q_budget=lookup[(t, tuple(sorted((u, v))))]["q_rate"], p_budget=lookup[(t, tuple(sorted((u, v))))]["p_rate"]) for u, v in G.edges()}
        for flow in order:
            cs = [evaluate_path(p, t, lookup, residual, flow, params) for p in cands[flow["flow_id"]]]
            for c in cs:
                c["method"] = method
                c["selection_score"] = selection_score(c, method, rng, ctrl)
            # service-state hierarchy shared by all methods; the method decides WITHIN a state
            secure = [c for c in cs if c["secure_feasible"]]
            best_effort = [c for c in cs if c["resource_feasible"] and not c["secure_feasible"]]
            degraded = [c for c in cs if c["path_available"] and not c["resource_feasible"]]
            if method == "shortest_path_pqc":
                # security-agnostic: picks the fastest resource-feasible path regardless of secure feasibility
                pool = secure + best_effort
                if pool:
                    sel = max(pool, key=lambda x: x["selection_score"])
                    sel["selection_status"] = "secure_allocated" if sel["secure_feasible"] else "best_effort_allocated"
                    sel["allocated"] = True
                    consume(sel["path"], residual, flow, G)
                elif degraded:
                    sel = max(degraded, key=lambda x: x["selection_score"]); sel["selection_status"] = "degraded_unallocated"; sel["allocated"] = False
                else:
                    sel = blocked_row(flow, t, method)
            elif secure:
                sel = max(secure, key=lambda x: x["selection_score"]); sel["selection_status"] = "secure_allocated"; sel["allocated"] = True
                consume(sel["path"], residual, flow, G)
            elif best_effort:
                sel = max(best_effort, key=lambda x: x["selection_score"]); sel["selection_status"] = "best_effort_allocated"; sel["allocated"] = True
                consume(sel["path"], residual, flow, G)
            elif degraded:
                sel = max(degraded, key=lambda x: x["selection_score"]); sel["selection_status"] = "degraded_unallocated"; sel["allocated"] = False
            else:
                sel = blocked_row(flow, t, method)
            rows.append(sel)
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# System-level indices (Eqs. (11), (24)) and regimes
# -----------------------------------------------------------------------------
def regime_thresholds(flows, params):
    sci_full = len(flows) * params["m_ref"] / sum(f["security_req"] for f in flows)
    return params["sparse_fraction"] * sci_full, params["secure_fraction"] * sci_full


def classify_regime(sci, thr):
    lo, hi = thr
    return "low_reserve" if sci < lo else ("intermediate_reserve" if sci < hi else "high_reserve")


def time_summary(alloc, flows, params):
    sreq = sum(f["security_req"] for f in flows)
    thr = regime_thresholds(flows, params)
    g = alloc.groupby("time")
    df = g.agg(secure_feasible_ratio=("secure_feasible", "mean"), allocated_ratio=("allocated", "mean"),
               mean_security_margin=("security_margin", "mean"), mean_q_share=("q_share", "mean"), mean_risk=("risk_path", "mean"),
               mean_security_level=("security_level", "mean"), mean_functional_score=("functional_score", "mean"),
               mean_latency=("latency_path", "mean"), mean_free_energy=("free_energy", "mean"),
               n_secure=("selection_status", lambda s: (s == "secure_allocated").sum()),
               n_best_effort=("selection_status", lambda s: (s == "best_effort_allocated").sum()),
               n_degraded=("selection_status", lambda s: (s == "degraded_unallocated").sum()),
               n_blocked=("selection_status", lambda s: (s == "blocked").sum())).reset_index()
    df["SCI"] = g["security_margin"].apply(lambda m: np.clip(m, 0, None).sum()).values / sreq
    df["SDI"] = g["security_margin"].apply(lambda m: np.clip(-m, 0, None).sum()).values / sreq
    df["signed_margin"] = df["SCI"] - df["SDI"]
    df["regime"] = df["SCI"].apply(lambda s: classify_regime(s, thr))
    return df


def method_summary(alloc, ts):
    return dict(secure_feasible_ratio=alloc["secure_feasible"].mean(), allocated_ratio=alloc["allocated"].mean(),
                avg_security_margin=alloc["security_margin"].mean(), avg_functional_score=alloc.loc[alloc["allocated"], "functional_score"].mean(),
                avg_q_share=alloc["q_share"].mean(), avg_risk=alloc["risk_path"].mean(), avg_latency=alloc["latency_path"].mean(),
                avg_SCI=ts["SCI"].mean(), avg_SDI=ts["SDI"].mean(), avg_free_energy=alloc["free_energy"].mean())


# -----------------------------------------------------------------------------
# Experiment driver
# -----------------------------------------------------------------------------
def run_experiment(seed, scenario, params, methods=METHODS, ctrl=None):
    """One seed, one scenario, all methods. Returns (alloc_df, time_summary_df, link_df, G)."""
    rng = np.random.default_rng(seed)
    G = build_graph(rng)
    link_df = apply_scenario(simulate_link_states(G, rng, params), scenario, params)
    lookup = {(int(r["time"]), tuple(sorted((r["u"], r["v"])))): r for r in link_df.to_dict("records")}
    flows = scenario_flows(scenario)
    cands = candidate_paths(G, flows, params)
    allocs, tss = [], []
    for m in methods:
        a = run_method(m, G, lookup, flows, cands, params, rng=np.random.default_rng(seed + 7919), ctrl=ctrl)
        ts = time_summary(a, flows, params)
        a["seed"], a["scenario"] = seed, scenario
        ts["seed"], ts["scenario"], ts["method"] = seed, scenario, m
        allocs.append(a); tss.append(ts)
    return pd.concat(allocs, ignore_index=True), pd.concat(tss, ignore_index=True), link_df, G


import types as _types
qm = _types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith("__")})


## 2. Experiments
Each section writes its tables (CSV) and figures (PNG) under the output root.

In [3]:
# =============================================================================
# Experiment driver: nominal run, method comparison, ablation, repeated seeds,
# stress scenarios, parameter sensitivity, prospective SCI evaluation.
# Produces every table (CSV) and figure (PNG) of the manuscript.
# =============================================================================
import os, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx


ROOT = Path(OUTPUT_ROOT)
FIG, TAB, OTH = ROOT / "figures", ROOT / "tables", ROOT / "others"
for d in (FIG, TAB, OTH): d.mkdir(parents=True, exist_ok=True)

REF_SEED = 42
N_SEEDS = int(N_SEEDS_ENV)
SEEDS = list(range(1, N_SEEDS + 1))
P = dict(qm.PARAMS)
THR = qm.regime_thresholds(qm.FLOWS, P)

# ---- plotting defaults (categorical palette validated for CVD separation) ----
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#7a5bd1"]
MCOL = dict(zip(qm.METHODS, PAL))
FCOL = dict(zip(["F1", "F2", "F3", "F4", "F5"], PAL))
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "legend.frameon": False})
def save(name):
    plt.tight_layout(); plt.savefig(FIG / name, bbox_inches="tight"); plt.close()
def lab(m): return qm.METHOD_LABELS[m]

t0 = time.time()
json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in P.items()} | {"thresholds": THR, "seeds": SEEDS, "ref_seed": REF_SEED}, open(OTH / "experiment_config.json", "w"), indent=2)


### A. Reference run (seed 42, nominal)

In [4]:
# =============================================================================
# A. Reference run (seed 42, nominal, all methods)
# =============================================================================
alloc0, ts0, link0, G0 = qm.run_experiment(REF_SEED, "nominal", P)
alloc0.to_csv(TAB / "reference_allocation_decisions.csv", index=False)
ts0.to_csv(TAB / "reference_time_summary.csv", index=False)
link0.to_csv(TAB / "reference_link_states.csv", index=False)
prop = alloc0[alloc0.method == "proposed_hybrid"].copy()
tprop = ts0[ts0.method == "proposed_hybrid"].copy()

# fig1 topology
plt.figure(figsize=(7, 5.5))
pos = {n: G0.nodes[n]["pos"] for n in G0.nodes()}
role_col = {"satellite": PAL[0], "aerial": PAL[2], "gateway": PAL[3], "base_station": PAL[1], "edge_cloud": PAL[4]}
nx.draw_networkx_edges(G0, pos, edge_color=["#2a78d6" if G0[u][v]["supports_qkd"] else "#9a9a9a" for u, v in G0.edges()],
                       width=[2.2 if G0[u][v]["supports_qkd"] else 1.4 for u, v in G0.edges()], style=["solid" if G0[u][v]["supports_qkd"] else "dashed" for u, v in G0.edges()])
nx.draw_networkx_nodes(G0, pos, node_color=[role_col[G0.nodes[n]["role"]] for n in G0.nodes()], node_size=900, edgecolors="white", linewidths=1.5)
nx.draw_networkx_labels(G0, pos, font_size=8, font_color="white", font_weight="bold")
for r, c in role_col.items(): plt.scatter([], [], c=c, s=80, label=r.replace("_", " "))
plt.plot([], [], c="#2a78d6", lw=2.2, label="QKD-capable link"); plt.plot([], [], c="#9a9a9a", lw=1.4, ls="--", label="PQC-only link")
plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8); plt.axis("off"); plt.title("Role-aware STIN topology (10 nodes, 18 links)")
save("fig2_role_aware_topology.png")

# fig2 allocation outcomes over time
st_cols = {"secure_allocated": PAL[2], "best_effort_allocated": PAL[3], "degraded_unallocated": PAL[1], "blocked": "#555555"}
cnt = prop.groupby(["time", "selection_status"]).size().unstack(fill_value=0).reindex(columns=list(st_cols), fill_value=0)
plt.figure(); bottom = np.zeros(len(cnt))
for s_, c in st_cols.items():
    plt.bar(cnt.index, cnt[s_], bottom=bottom, color=c, label=s_.replace("_", " "), width=0.8, edgecolor="white", linewidth=0.8); bottom += cnt[s_].values
plt.xlabel("Time step"); plt.ylabel("Number of flows"); plt.title("Allocation outcomes over time (proposed hybrid)"); plt.legend(ncol=2, fontsize=8)
save("fig3_allocation_outcomes_over_time.png")

# fig3 SCI and SDI over time with regimes
plt.figure()
plt.axhspan(0, THR[0], color=PAL[1], alpha=0.08); plt.axhspan(THR[0], THR[1], color=PAL[3], alpha=0.08); plt.axhspan(THR[1], 1, color=PAL[2], alpha=0.08)
plt.plot(tprop.time, tprop.SCI, color=PAL[0], lw=2, marker="o", ms=4, label="SCI")
plt.plot(tprop.time, tprop.SDI, color=PAL[1], lw=2, marker="s", ms=4, label="SDI")
plt.axhline(THR[1], color="#777", lw=1, ls=":"); plt.axhline(THR[0], color="#777", lw=1, ls=":")
plt.text(23.5, THR[1] + 0.01, r"$\theta_{high}$", ha="right", fontsize=8, color="#555"); plt.text(23.5, THR[0] + 0.01, r"$\theta_{low}$", ha="right", fontsize=8, color="#555")
plt.ylim(0, max(0.9, tprop.SCI.max() + 0.1)); plt.xlabel("Time step"); plt.ylabel("Index value"); plt.title("SCI and SDI over time (proposed hybrid)"); plt.legend()
save("fig5_sci_sdi_over_time.png")

# fig4 secure-feasible and allocated ratio
plt.figure()
plt.plot(tprop.time, tprop.secure_feasible_ratio, color=PAL[0], lw=2, marker="o", ms=4, label="Secure-feasible ratio")
plt.plot(tprop.time, tprop.allocated_ratio, color=PAL[2], lw=2, marker="s", ms=4, label="Allocated ratio")
plt.ylim(-0.02, 1.05); plt.xlabel("Time step"); plt.ylabel("Ratio"); plt.title("Secure feasibility and allocation over time (proposed hybrid)"); plt.legend()
save("fig4_secure_feasibility_and_allocation.png")

# fig5 hybrid contribution dynamics
hc = prop[prop.allocated].groupby("time")[["Q", "C", "q_share"]].mean()
plt.figure()
plt.plot(hc.index, hc.q_share, color=PAL[0], lw=2, marker="o", ms=4, label="Mean QKD share $Q/(Q+C)$")
plt.plot(hc.index, 1 - hc.q_share, color=PAL[1], lw=2, marker="s", ms=4, label="Mean PQC share $C/(Q+C)$")
plt.ylim(0, 1); plt.xlabel("Time step"); plt.ylabel("Share"); plt.title("Hybrid QKD versus PQC contribution over time (proposed hybrid)"); plt.legend()
save("fig7_hybrid_contribution_dynamics.png")

# fig6 margin vs latency
plt.figure()
for f, g in prop[prop.allocated].groupby("flow_id"):
    plt.scatter(g.latency_path, g.security_margin, color=FCOL[f], s=28, alpha=0.85, label=f, edgecolors="white", linewidths=0.5)
plt.axhline(0, color="#555", lw=1); plt.xlabel("Path latency (ms)"); plt.ylabel("Security margin $M_p$"); plt.title("Security margin versus latency by flow (proposed hybrid)"); plt.legend(title="Flow")
save("fig8_security_margin_vs_latency.png")

# per-flow summary table (Table 5)
flow_sum = prop.groupby("flow_id").agg(secure_feasible_ratio=("secure_feasible", "mean"), allocated_ratio=("allocated", "mean"),
                                       avg_security_margin=("security_margin", "mean"), avg_functional_score=("functional_score", "mean"),
                                       avg_q_share=("q_share", "mean"), avg_latency=("latency_path", "mean")).reset_index()
flow_sum.to_csv(TAB / "summary_by_flow.csv", index=False)

# fig7 per-flow feasibility
plt.figure(); x = np.arange(len(flow_sum)); w = 0.38
plt.bar(x - w / 2, flow_sum.secure_feasible_ratio, w, color=PAL[0], label="Secure-feasible ratio")
plt.bar(x + w / 2, flow_sum.allocated_ratio, w, color=PAL[2], label="Allocated ratio")
plt.xticks(x, flow_sum.flow_id); plt.ylim(0, 1.05); plt.ylabel("Ratio"); plt.title("Per-flow secure feasibility and allocation (proposed hybrid)"); plt.legend()
save("fig9_per_flow_feasibility_and_allocation.png")

# path stability (Table 6) + fig8
ps = []
for f, g in prop.sort_values("time").groupby("flow_id"):
    ps.append(dict(flow_id=f, path_switch_rate=(g.path.values[1:] != g.path.values[:-1]).mean(),
                   status_switch_rate=(g.selection_status.values[1:] != g.selection_status.values[:-1]).mean(),
                   unique_paths=g.loc[g.path != "BLOCKED", "path"].nunique(), blocked_fraction=(g.selection_status == "blocked").mean(),
                   secure_fraction=(g.selection_status == "secure_allocated").mean()))
ps = pd.DataFrame(ps); ps.to_csv(TAB / "path_stability_summary.csv", index=False)
plt.figure(); plt.bar(x - w / 2, ps.path_switch_rate, w, color=PAL[0], label="Path-switch rate"); plt.bar(x + w / 2, ps.status_switch_rate, w, color=PAL[1], label="Status-switch rate")
plt.xticks(x, ps.flow_id); plt.ylabel("Rate"); plt.title("Path and status switching by flow (proposed hybrid)"); plt.legend()
save("fig10_path_and_status_switching.png")


### B. Method comparison on the reference run

In [5]:
# =============================================================================
# B. Method comparison on the reference run (Table 3, figs 9-15, 18, 19)
# =============================================================================
ms_rows = []
for m in qm.METHODS:
    a, ts = alloc0[alloc0.method == m], ts0[ts0.method == m]
    ms_rows.append(dict(method=m, **qm.method_summary(a, ts)))
ms = pd.DataFrame(ms_rows); ms.to_csv(TAB / "comparison_method_summary.csv", index=False)

def grouped_methods(df, cols, labels, fname, title, ylabel, hline=False):
    plt.figure(figsize=(8, 4.5)); n = len(cols); xx = np.arange(len(df)); w = 0.8 / n
    for i, (c, l) in enumerate(zip(cols, labels)):
        plt.bar(xx + (i - (n - 1) / 2) * w, df[c], w, label=l, color=PAL[i], edgecolor="white", linewidth=0.6)
    if hline: plt.axhline(0, color="#555", lw=1)
    plt.xticks(xx, [lab(m) for m in df.method], rotation=20, ha="right"); plt.ylabel(ylabel); plt.title(title); plt.legend(fontsize=8)
    save(fname)
grouped_methods(ms, ["secure_feasible_ratio", "allocated_ratio", "avg_SCI"], ["Secure-feasible ratio", "Allocated ratio", "Average SCI"],
                "fig11_method_overall_comparison.png", "Overall comparison of allocation methods (reference run)", "Value")
plt.figure()
for m in qm.METHODS:
    t = ts0[ts0.method == m]; plt.plot(t.time, t.SCI, color=MCOL[m], lw=2 if m == "proposed_hybrid" else 1.3, marker="o" if m == "proposed_hybrid" else None, ms=3, label=lab(m))
plt.xlabel("Time step"); plt.ylabel("SCI"); plt.title("SCI over time by method (reference run)"); plt.legend(fontsize=8, ncol=2)
save("fig12_sci_over_time_by_method.png")
plt.figure()
for m in qm.METHODS:
    t = ts0[ts0.method == m]; plt.plot(t.time, t.secure_feasible_ratio, color=MCOL[m], lw=2 if m == "proposed_hybrid" else 1.3, label=lab(m))
plt.ylim(-0.02, 1.05); plt.xlabel("Time step"); plt.ylabel("Secure-feasible ratio"); plt.title("Secure-feasible ratio over time by method"); plt.legend(fontsize=8, ncol=2)
save("fig13_secure_feasible_ratio_by_method.png")
comp = alloc0.groupby(["method", "selection_status"]).size().unstack(fill_value=0).reindex(index=qm.METHODS, columns=list(st_cols), fill_value=0)
comp = comp.div(comp.sum(axis=1), axis=0); comp.to_csv(TAB / "comparison_method_status_summary.csv")
plt.figure(figsize=(8, 4.5)); bottom = np.zeros(len(comp))
for s_, c in st_cols.items():
    plt.bar(range(len(comp)), comp[s_], bottom=bottom, color=c, label=s_.replace("_", " "), edgecolor="white"); bottom += comp[s_].values
plt.xticks(range(len(comp)), [lab(m) for m in comp.index], rotation=20, ha="right"); plt.ylabel("Fraction of (flow, time) pairs"); plt.title("Allocation-state composition by method"); plt.legend(fontsize=8, ncol=2)
save("fig14_allocation_state_composition_by_method.png")
pfm = alloc0.groupby(["flow_id", "method"]).security_margin.mean().unstack().reindex(columns=qm.METHODS); pfm.to_csv(TAB / "comparison_method_flow_margin.csv")
plt.figure(figsize=(8, 4.5)); n = len(qm.METHODS); xx = np.arange(len(pfm)); w = 0.8 / n
for i, m in enumerate(qm.METHODS):
    plt.bar(xx + (i - (n - 1) / 2) * w, pfm[m], w, color=MCOL[m], label=lab(m), edgecolor="white", linewidth=0.5)
plt.axhline(0, color="#555", lw=1); plt.xticks(xx, pfm.index); plt.ylabel("Average security margin"); plt.title("Per-flow security margin by method"); plt.legend(fontsize=7, ncol=2)
save("fig15_per_flow_security_margin_by_method.png")
plt.figure()
for m in qm.METHODS:
    plt.scatter(ms.loc[ms.method == m, "avg_q_share"], ms.loc[ms.method == m, "avg_latency"], color=MCOL[m], s=90, label=lab(m), edgecolors="white")
plt.xlabel("Average quantum share"); plt.ylabel("Average latency (ms)"); plt.title("Latency versus quantum-share participation by method"); plt.legend(fontsize=8)
save("fig16_latency_vs_quantum_share_by_method.png")
plt.figure(); plt.bar([lab(m) for m in ms.method], ms.avg_functional_score, color=[MCOL[m] for m in ms.method]); plt.xticks(rotation=20, ha="right")
plt.ylabel("Average functional score $U_p$"); plt.title("Average functional score by method"); save("fig17_functional_score_by_method.png")

def phase_plot(t_df, title, fname, by_method=False):
    plt.figure(figsize=(7, 5))
    plt.axhspan(0, THR[0], color=PAL[1], alpha=0.07); plt.axhspan(THR[0], THR[1], color=PAL[3], alpha=0.07); plt.axhspan(THR[1], 1.2, color=PAL[2], alpha=0.07)
    if by_method:
        for m in qm.METHODS:
            g = t_df[t_df.method == m]; plt.scatter(g.mean_risk, g.SCI, s=30 + 120 * g.secure_feasible_ratio, color=MCOL[m], alpha=0.75, label=lab(m), edgecolors="white", linewidths=0.5)
        plt.legend(fontsize=8)
    else:
        sc = plt.scatter(t_df.mean_risk, t_df.SCI, s=30 + 120 * t_df.secure_feasible_ratio, c=t_df.mean_q_share, cmap="Blues", vmin=0, vmax=1, edgecolors="#333", linewidths=0.6)
        plt.colorbar(sc, label="Mean quantum share")
    plt.xlabel("Mean path risk $R^{risk}$"); plt.ylabel("SCI"); plt.ylim(0, max(0.9, t_df.SCI.max() + 0.1)); plt.title(title)
    for y, s_ in [(THR[0] / 2, "low reserve"), ((THR[0] + THR[1]) / 2, "intermediate reserve"), (THR[1] + 0.08, "high reserve")]:
        plt.text(plt.xlim()[1], y, s_, ha="right", va="center", fontsize=8, color="#555")
    save(fname)
phase_plot(tprop, "Reserve-band diagram (proposed hybrid). Marker size: secure-feasible ratio", "fig6_regime_diagram_proposed.png")
phase_plot(ts0, "Reserve-band diagram by method", "fig18_regime_diagram_by_method.png", by_method=True)


### C. Ablation of the controller signals

In [6]:
# =============================================================================
# C. Ablation (Table 9, figs 16-17): reference run and pooled over seeds
# =============================================================================
ABL = {"Full model": {}, "No quantum signal": {"use_quantum_signal": False}, "No risk penalty": {"use_risk_penalty": False},
       "No trust signal": {"use_trust_signal": False}, "No entropy term": {"use_entropy_term": False}}
abl_rows, abl_ts = [], {}
for name, ov in ABL.items():
    Ca = dict(P, **ov)   # controller parameters only; the state model (S_p, feasibility) is unchanged
    a, ts, _, _ = qm.run_experiment(REF_SEED, "nominal", P, methods=["proposed_hybrid"], ctrl=Ca)
    abl_ts[name] = ts
    pooled = []
    for sd in SEEDS:
        a2, ts2, _, _ = qm.run_experiment(sd, "nominal", P, methods=["proposed_hybrid"], ctrl=Ca); pooled.append(dict(qm.method_summary(a2, ts2)))
    pooled = pd.DataFrame(pooled)
    r = dict(variant=name, **{k: v for k, v in qm.method_summary(a, ts).items()})
    r.update({f"pooled_{k}": pooled[k].mean() for k in ["secure_feasible_ratio", "allocated_ratio", "avg_security_margin", "avg_SCI", "avg_functional_score"]})
    r.update({f"pooled_sd_{k}": pooled[k].std() for k in ["secure_feasible_ratio", "avg_SCI"]})
    abl_rows.append(r)
abl = pd.DataFrame(abl_rows); abl.to_csv(TAB / "ablation_summary.csv", index=False)
plt.figure(figsize=(8, 4.5)); xx = np.arange(len(abl)); w = 0.26
for i, (c, l) in enumerate(zip(["pooled_secure_feasible_ratio", "pooled_avg_security_margin", "pooled_avg_SCI"], ["Secure-feasible ratio", "Avg. security margin", "Avg. SCI"])):
    plt.bar(xx + (i - 1) * w, abl[c], w, color=PAL[i], label=l, edgecolor="white")
plt.axhline(0, color="#555", lw=1); plt.xticks(xx, abl.variant, rotation=15, ha="right"); plt.title(f"Ablation (nominal, mean over {N_SEEDS} seeds)"); plt.legend(fontsize=8)
save("fig21_ablation_overall_comparison.png")
plt.figure()
for i, (name, ts) in enumerate(abl_ts.items()):
    plt.plot(ts.time, ts.SCI, color=PAL[i], lw=2.2 if name == "Full model" else 1.3, marker="o" if name == "Full model" else None, ms=3, label=name)
plt.xlabel("Time step"); plt.ylabel("SCI"); plt.title("SCI over time under ablation variants (reference run)"); plt.legend(fontsize=8)
save("fig22_ablation_sci_over_time.png")


### D. Repeated seeds and stress scenarios

In [7]:
# =============================================================================
# D. Repeated seeds x stress scenarios (Tables 7-8, fig20)
# =============================================================================
all_ts, all_ms = [], []
for sc in qm.SCENARIOS:
    for sd in SEEDS:
        a, ts, _, _ = qm.run_experiment(sd, sc, P)
        all_ts.append(ts)
        for m in qm.METHODS:
            all_ms.append(dict(scenario=sc, seed=sd, method=m, **qm.method_summary(a[a.method == m], ts[ts.method == m])))
all_ts = pd.concat(all_ts, ignore_index=True); all_ms = pd.DataFrame(all_ms)
all_ts.to_csv(TAB / "seed_scenario_time_summary.csv", index=False); all_ms.to_csv(TAB / "seed_scenario_method_summary.csv", index=False)
metrics = ["secure_feasible_ratio", "allocated_ratio", "avg_security_margin", "avg_SCI", "avg_SDI", "avg_functional_score", "avg_latency"]
seed_sum = all_ms[all_ms.scenario == "nominal"].groupby("method")[metrics].agg(["mean", "std"]).reindex(qm.METHODS)
seed_sum.columns = [f"{a}_{b}" for a, b in seed_sum.columns]
wins = all_ms[all_ms.scenario == "nominal"].pivot(index="seed", columns="method", values="avg_functional_score")
seed_sum["best_functional_fraction"] = (wins.idxmax(axis=1).value_counts() / len(wins)).reindex(qm.METHODS).fillna(0)
wins2 = all_ms[all_ms.scenario == "nominal"].pivot(index="seed", columns="method", values="avg_SCI")
seed_sum["best_SCI_fraction"] = (wins2.idxmax(axis=1).value_counts() / len(wins2)).reindex(qm.METHODS).fillna(0)
seed_sum.reset_index().to_csv(TAB / "nominal_seed_summary.csv", index=False)
scen_sum = all_ms.groupby(["scenario", "method"])[metrics].mean().reset_index(); scen_sum.to_csv(TAB / "scenario_method_summary.csv", index=False)
regime_frac = all_ts.groupby(["scenario", "method"]).regime.value_counts(normalize=True).unstack(fill_value=0).reset_index(); regime_frac.to_csv(TAB / "scenario_regime_fractions.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=False)
scs = list(qm.SCENARIOS); xx = np.arange(len(scs)); n = len(qm.METHODS); w = 0.8 / n
for ax, (c, ttl) in zip(axes, [("avg_SCI", "Average SCI"), ("avg_SDI", "Average SDI"), ("avg_security_margin", "Average security margin")]):
    for i, m in enumerate(qm.METHODS):
        v = scen_sum[scen_sum.method == m].set_index("scenario").reindex(scs)[c]
        ax.bar(xx + (i - (n - 1) / 2) * w, v, w, color=MCOL[m], label=lab(m), edgecolor="white", linewidth=0.4)
    ax.set_xticks(xx); ax.set_xticklabels([s.replace("_", "\n") for s in scs], fontsize=8); ax.set_title(ttl); ax.axhline(0, color="#555", lw=0.8)
axes[0].legend(fontsize=7, ncol=2); fig.suptitle(f"Method performance under stress scenarios (mean over {N_SEEDS} seeds)")
plt.tight_layout(); plt.savefig(FIG / "fig19_scenario_summary.png", bbox_inches="tight"); plt.close()


### E. One-at-a-time parameter sensitivity

In [8]:
# =============================================================================
# E. One-at-a-time parameter sensitivity (fig21)
# =============================================================================
SWEEPS = {"alpha": [0.35, 0.45, 0.55, 0.65, 0.75], "lambda_H": [0.0, 0.125, 0.25, 0.5, 1.0], "lambda_R": [0.0, 0.225, 0.45, 0.9, 1.35],
          "beta_trust": [0.0, 0.11, 0.22, 0.33, 0.44], "w_ell": [0.0, 0.15, 0.30, 0.60, 0.90],
          "beta_risk": [(0.45, 0.30, 0.25), (0.70, 0.15, 0.15), (0.15, 0.70, 0.15), (0.15, 0.15, 0.70), (1 / 3, 1 / 3, 1 / 3)],
          "thresholds": [-0.25, 0.0, 0.25]}
base_dec, sens_rows = {}, []
for sd in SEEDS:
    a, ts, _, _ = qm.run_experiment(sd, "nominal", P, methods=["proposed_hybrid"]); base_dec[sd] = (a, ts)
for pname, vals in SWEEPS.items():
    for v in vals:
        MODEL_PARAMS = {"alpha", "beta_trust"}   # change the state definition S_p as well as the controller
        if pname == "thresholds":
            Pa, Ca = dict(P, secure_fraction=P["secure_fraction"] * (1 + v), sparse_fraction=P["sparse_fraction"] * (1 + v)), None
        elif pname in MODEL_PARAMS:
            Pa, Ca = dict(P, **{pname: v}), None
        else:
            Pa, Ca = P, dict(P, **{pname: v})
        ch, rec = [], []
        for sd in SEEDS:
            if pname == "thresholds":
                a, ts = base_dec[sd]; ts = qm.time_summary(a, qm.scenario_flows("nominal"), Pa)
            else:
                a, ts, _, _ = qm.run_experiment(sd, "nominal", Pa, methods=["proposed_hybrid"], ctrl=Ca)
            b = base_dec[sd][0]
            ch.append((a.path.values != b.path.values).mean())
            rl = (ts.regime.values != base_dec[sd][1].regime.values).mean()
            rec.append(dict(qm.method_summary(a, ts), decision_change=ch[-1], regime_label_change=rl))
        rec = pd.DataFrame(rec)
        sens_rows.append(dict(parameter=pname, value=str(v), scope="model+controller" if pname in MODEL_PARAMS else ("regime labels" if pname == "thresholds" else "controller"), secure_feasible_ratio=rec.secure_feasible_ratio.mean(), avg_SCI=rec.avg_SCI.mean(),
                              avg_functional_score=rec.avg_functional_score.mean(), avg_security_margin=rec.avg_security_margin.mean(),
                              decision_change_fraction=rec.decision_change.mean(), regime_label_change_fraction=rec.regime_label_change.mean()))
sens = pd.DataFrame(sens_rows); sens.to_csv(TAB / "parameter_sensitivity.csv", index=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5)); axes = axes.ravel()
for ax, pname in zip(axes, SWEEPS):
    g = sens[sens.parameter == pname]; xx = np.arange(len(g))
    ax.plot(xx, g.secure_feasible_ratio, color=PAL[0], marker="o", label="Secure-feasible ratio")
    ax.plot(xx, g.avg_SCI, color=PAL[2], marker="s", label="Avg. SCI")
    ax.plot(xx, g.decision_change_fraction if pname != "thresholds" else g.regime_label_change_fraction, color=PAL[1], marker="^", label="Decisions changed" if pname != "thresholds" else "Regime labels changed")
    ax.set_xticks(xx); ax.set_xticklabels([v if pname != "beta_risk" else "(" + ",".join(f"{x:.2f}" for x in eval(v)) + ")" for v in g.value], fontsize=7, rotation=25)
    ax.set_title({"alpha": r"$\alpha$", "lambda_H": r"$\lambda_H$", "lambda_R": r"$\lambda_R$", "beta_trust": r"$\beta_\tau$", "w_ell": r"$w_\ell$", "beta_risk": r"$(\beta_1,\beta_2,\beta_3)$", "thresholds": r"$\theta$ scaling"}[pname]); ax.set_ylim(0, 1)
    if pname == "alpha": ax.legend(fontsize=7)
axes[-1].axis("off"); fig.suptitle(f"One-at-a-time parameter sensitivity of the proposed controller (nominal, mean over {N_SEEDS} seeds)")
plt.tight_layout(); plt.savefig(FIG / "fig20_parameter_sensitivity.png", bbox_inches="tight"); plt.close()


### F. Prospective evaluation of the SCI

In [9]:
# =============================================================================
# F. Prospective evaluation of SCI (Table 4)
# =============================================================================
def auc_score(scores, labels):
    scores, labels = np.asarray(scores, float), np.asarray(labels, bool)
    pos, neg = scores[labels], scores[~labels]
    if len(pos) == 0 or len(neg) == 0: return np.nan
    # Mann-Whitney: probability that a positive scores higher than a negative (ties count 1/2)
    return float((np.sum(pos[:, None] > neg[None, :]) + 0.5 * np.sum(pos[:, None] == neg[None, :])) / (len(pos) * len(neg)))

tp_prop = all_ts[all_ts.method == "proposed_hybrid"]
pr_rows = []
LOW = 0.4  # failure state: at most 2 of the 5 flows secure-feasible
for k in (1, 2, 3):
    lbl, sci, sdi, sgn, pers = [], [], [], [], []
    for (sc, sd), g in tp_prop.groupby(["scenario", "seed"]):
        g = g.sort_values("time"); s = g.secure_feasible_ratio.values; x = g.SCI.values; y = g.SDI.values; m = g.signed_margin.values
        for t in range(len(s) - k):
            if s[t] > LOW:   # not yet in the failure state: does the indicator anticipate entry at t+k?
                lbl.append(s[t + k] <= LOW); sci.append(-x[t]); sdi.append(y[t]); sgn.append(-m[t]); pers.append(-s[t])
    lbl, sci, sdi, sgn, pers = map(np.array, (lbl, sci, sdi, sgn, pers))
    # operating point: threshold on SCI maximising Youden's J over the pooled sample
    cands_thr = np.unique(-sci); best = None
    for th in cands_thr:
        al = (-sci) < th; tp = np.sum(al & lbl); fp = np.sum(al & ~lbl); fn = np.sum(~al & lbl); tn = np.sum(~al & ~lbl)
        J = tp / max(tp + fn, 1) - fp / max(fp + tn, 1)
        if best is None or J > best[0]: best = (J, th, tp, fp, fn, tn)
    J, th, tp, fp, fn, tn = best
    # lead time: for each failure onset, earliest alarm (SCI < th) within the 3 preceding steps
    det, lead = [], []
    for (sc, sd), g in tp_prop.groupby(["scenario", "seed"]):
        g = g.sort_values("time"); s = g.secure_feasible_ratio.values; x = g.SCI.values
        for t in range(1, len(s)):
            if s[t] <= LOW and s[t - 1] > LOW:
                al = [u for u in range(max(0, t - 3), t) if x[u] < th]
                det.append(bool(al)); lead.append(t - min(al) if al else 0)
    pr_rows.append(dict(horizon=k, n_samples=len(lbl), event_rate=lbl.mean(), alarm_threshold_SCI=th, precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                        false_alarm_rate=fp / max(fp + tn, 1), auc_SCI=auc_score(sci, lbl), auc_SDI=auc_score(sdi, lbl), auc_signed_margin=auc_score(sgn, lbl),
                        auc_persistence_baseline=auc_score(pers, lbl), onset_events=len(det), onset_detection_rate=np.mean(det) if det else np.nan,
                        mean_lead_time_detected=np.mean([l for l in lead if l > 0]) if any(l > 0 for l in lead) else np.nan))
prosp = pd.DataFrame(pr_rows); prosp.to_csv(TAB / "sci_prospective_evaluation.csv", index=False)



### G. Held-out validation of the SCI alarm and paired method comparisons

In [10]:
# =============================================================================
# G. Held-out validation of the SCI alarm and paired method comparisons
# =============================================================================
rng_b = np.random.default_rng(2026)
DEV, TEST = SEEDS[:2 * len(SEEDS) // 3], SEEDS[2 * len(SEEDS) // 3:]
def pooled_samples(df, seeds, k):
    rows = []
    for (sc, sd), g in df[df.seed.isin(seeds)].groupby(["scenario", "seed"]):
        g = g.sort_values("time"); s = g.secure_feasible_ratio.values; x = g.SCI.values
        for t in range(len(s) - k):
            if s[t] > LOW:   # at risk: not yet in the failure state at t
                rows.append(dict(scenario=sc, seed=sd, y=bool(s[t + k] <= LOW), sci=x[t], persist=s[t], minus_margin=-g.signed_margin.values[t]))
    return pd.DataFrame(rows)
def youden_threshold(d):
    best = None
    for th in np.unique(d.sci):
        al = d.sci < th; tp = (al & d.y).sum(); fp = (al & ~d.y).sum(); fn = (~al & d.y).sum(); tn = (~al & ~d.y).sum()
        J = tp / max(tp + fn, 1) - fp / max(fp + tn, 1)
        if best is None or J > best[0]: best = (J, th)
    return best[1]
def alarm_stats(d, th):
    al = d.sci < th; tp = (al & d.y).sum(); fp = (al & ~d.y).sum(); fn = (~al & d.y).sum(); tn = (~al & ~d.y).sum()
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1), false_alarm_rate=fp / max(fp + tn, 1))
def cluster_boot(d, fn, B=1000):
    seeds = d.seed.unique(); groups = {s: d[d.seed == s] for s in seeds}; vals = []
    for _ in range(B):
        samp = pd.concat([groups[s] for s in rng_b.choice(seeds, len(seeds), replace=True)])
        v = fn(samp)
        if not np.isnan(v): vals.append(v)
    return np.percentile(vals, [2.5, 97.5])
ho_rows, sc_rows = [], []
for k in (1, 2, 3):
    dev, test = pooled_samples(tp_prop, DEV, k), pooled_samples(tp_prop, TEST, k)
    th = youden_threshold(dev)
    st = alarm_stats(test, th)
    auc_t = auc_score(-test.sci, test.y); auc_p = auc_score(-test.persist, test.y); auc_m = auc_score(test.minus_margin, test.y)
    ci_auc = cluster_boot(test, lambda d: auc_score(-d.sci, d.y))
    ci_diff = cluster_boot(test, lambda d: auc_score(-d.sci, d.y) - auc_score(-d.persist, d.y))
    ci_prec = cluster_boot(test, lambda d: alarm_stats(d, th)["precision"])
    ci_rec = cluster_boot(test, lambda d: alarm_stats(d, th)["recall"])
    ho_rows.append(dict(horizon=k, dev_seeds=len(DEV), test_seeds=len(TEST), n_test=len(test), events_test=int(test.y.sum()), threshold_dev=th,
                        auc_SCI_test=auc_t, auc_SCI_ci_lo=ci_auc[0], auc_SCI_ci_hi=ci_auc[1], auc_persistence_test=auc_p, auc_signed_margin_test=auc_m,
                        auc_gain_ci_lo=ci_diff[0], auc_gain_ci_hi=ci_diff[1], precision_test=st["precision"], precision_ci_lo=ci_prec[0], precision_ci_hi=ci_prec[1],
                        recall_test=st["recall"], recall_ci_lo=ci_rec[0], recall_ci_hi=ci_rec[1], false_alarm_rate_test=st["false_alarm_rate"]))
    allk = pooled_samples(tp_prop, SEEDS, k)
    for sc, g in allk.groupby("scenario"):
        sc_rows.append(dict(horizon=k, scenario=sc, n=len(g), events=int(g.y.sum()), auc_SCI=auc_score(-g.sci, g.y), auc_persistence=auc_score(-g.persist, g.y)))
pd.DataFrame(ho_rows).to_csv(TAB / "sci_heldout_validation.csv", index=False)
pd.DataFrame(sc_rows).to_csv(TAB / "sci_auc_by_scenario.csv", index=False)

# paired seed-wise differences: proposed minus each alternative (seed = cluster; scenarios analysed separately)
pd_rows = []
for sc in qm.SCENARIOS:
    sub = all_ms[all_ms.scenario == sc]
    for m in qm.METHODS:
        if m == "proposed_hybrid": continue
        for met in ["avg_SCI", "avg_SDI", "secure_feasible_ratio", "allocated_ratio", "avg_latency"]:
            a = sub[sub.method == "proposed_hybrid"].set_index("seed")[met]; b = sub[sub.method == m].set_index("seed")[met]
            d = (a - b.reindex(a.index)).values
            boot = [rng_b.choice(d, len(d), replace=True).mean() for _ in range(2000)]
            lo, hi = np.percentile(boot, [2.5, 97.5])
            pd_rows.append(dict(scenario=sc, baseline=m, metric=met, mean_diff=d.mean(), ci_lo=lo, ci_hi=hi, n_seeds=len(d), wins=int((d > 0).sum())))
pd.DataFrame(pd_rows).to_csv(TAB / "paired_method_differences.csv", index=False)

# =============================================================================
# Manifest
# =============================================================================
man = {"elapsed_s": round(time.time() - t0, 1), "n_seeds": N_SEEDS, "figures": sorted(os.listdir(FIG)), "tables": sorted(os.listdir(TAB))}
json.dump(man, open(OTH / "run_manifest.json", "w"), indent=2)
print(json.dumps(man, indent=1))


{
 "elapsed_s": 750.3,
 "n_seeds": 30,
 "figures": [
  "fig10_path_and_status_switching.png",
  "fig11_method_overall_comparison.png",
  "fig12_sci_over_time_by_method.png",
  "fig13_secure_feasible_ratio_by_method.png",
  "fig14_allocation_state_composition_by_method.png",
  "fig15_per_flow_security_margin_by_method.png",
  "fig16_latency_vs_quantum_share_by_method.png",
  "fig17_functional_score_by_method.png",
  "fig18_regime_diagram_by_method.png",
  "fig19_scenario_summary.png",
  "fig20_parameter_sensitivity.png",
  "fig21_ablation_overall_comparison.png",
  "fig22_ablation_sci_over_time.png",
  "fig2_role_aware_topology.png",
  "fig3_allocation_outcomes_over_time.png",
  "fig4_secure_feasibility_and_allocation.png",
  "fig5_sci_sdi_over_time.png",
  "fig6_regime_diagram_proposed.png",
  "fig7_hybrid_contribution_dynamics.png",
  "fig8_security_margin_vs_latency.png",
  "fig9_per_flow_feasibility_and_allocation.png"
 ],
 "tables": [
  "ablation_summary.csv",
  "comparison_method_

In [11]:
# ---- Optional small backup to Google Drive (opt-in) ----------------------------
if IN_COLAB and BACKUP_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dst = f"/content/drive/MyDrive/Outputs/{PROJECT_NAME}"
    os.makedirs(dst, exist_ok=True)
    for sub in ("figures", "tables", "others"):
        shutil.copytree(os.path.join(OUTPUT_ROOT, sub), os.path.join(dst, sub), dirs_exist_ok=True)
    print("Backed up to", dst)
